# State diagram — Data validation

A short, reproducible check of the processed campaign. The original SLURM runs and processing jobs have already been validated; here we verify the local files and the identity of the realizations used in subsequent analyses. No physical conclusions are drawn in this notebook.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Works when Jupyter starts from the repository root or from state_diagram/.
workdir = Path.cwd().resolve()
root = workdir if (workdir / "processed").is_dir() else workdir / "state_diagram"
processed = root / "processed"
assert processed.is_dir(), f"Processed data not found at {processed}"

parquets = {
    "order": "order_parameters/order_parameters.parquet",
    "motion": "motion/motion.parquet",
    "deformation": "deformation/deformation.parquet",
    "overlap": "overlap/overlap.parquet",
    "cluster_time_series": "clusters/cluster_time_series.parquet",
    "cluster_snapshots": "clusters/cluster_snapshots.parquet",
    "cluster_size_counts": "clusters/cluster_size_counts.parquet",
    "cluster_raw": "clusters/cluster_raw.parquet",
    "local_summary": "local_order/local_order_summary.parquet",
    "local_grid": "local_order/local_order_grid.parquet",
}
missing = [relative for relative in parquets.values() if not (processed / relative).is_file()]
assert not missing, f"Missing Parquet files: {missing}"

file_summary = pd.DataFrame([
    {"dataset": name, "rows": pq.read_metadata(processed / relative).num_rows}
    for name, relative in parquets.items()
])
display(file_summary)
print(f"Located {len(parquets)} Parquet datasets.")

,dataset,rows
0,order,129280
1,motion,129280
2,deformation,51200
3,overlap,25600
4,cluster_time_series,107520
5,cluster_snapshots,18553
6,cluster_size_counts,133470
7,cluster_raw,1721210
8,local_summary,26880
9,local_grid,241786


Located 10 Parquet datasets.


## Consistency of the principal time-series datasets

Check the 20 parameter points, 64 realizations per point, and snapshot identifiers. Global order and motion should have identical snapshots; local-order summaries should match a subset of those snapshots (a coarser saving interval).

In [2]:
rho_values = (0.60, 0.75, 0.85, 0.90, 0.98)
kappa_values = (0.75, 1.50, 2.50, 4.00)
expected_points = {(rho, kappa) for rho in rho_values for kappa in kappa_values}
keys = ["rho", "kappa", "seed", "step"]

snapshots = {}
for dataset, expected_times in (("order", 101), ("motion", 101), ("local_summary", 21)):
    df = pd.read_parquet(processed / parquets[dataset], columns=keys + ["time"])
    assert not df.duplicated(keys).any(), f"Duplicate snapshots in {dataset}"
    assert np.allclose(df["time"], df["step"] * 0.01), f"Inconsistent time in {dataset}"
    points = set(map(tuple, df[["rho", "kappa"]].drop_duplicates().to_numpy()))
    assert points == expected_points, f"Unexpected parameter points in {dataset}"
    counts = df.groupby(["rho", "kappa", "seed"]).size()
    assert len(counts) == 1280 and counts.eq(expected_times).all(), f"Missing snapshots in {dataset}"
    snapshots[dataset] = pd.MultiIndex.from_frame(df[keys])
    print(f"{dataset}: {len(df):,} snapshots; {len(counts)} realizations; {len(points)} points — OK")

assert snapshots["order"].equals(snapshots["motion"]), "Order and motion snapshots differ"
assert snapshots["local_summary"].isin(snapshots["order"]).all(), "Local summary snapshots not in global order"
print("Snapshot identities consistent across order, motion, and local-order summary.")

order: 129,280 snapshots; 1280 realizations; 20 points — OK
motion: 129,280 snapshots; 1280 realizations; 20 points — OK
local_summary: 26,880 snapshots; 1280 realizations; 20 points — OK
Snapshot identities consistent across order, motion, and local-order summary.


The detailed cluster and spatial-grid datasets have different row structures and saving intervals. Their own processors already performed domain-specific checks; they are listed above but are not forced into the snapshot-table checks.